In [ ]:
!pip install datasets mne transformers torch torchvision

from huggingface_hub import snapshot_download
import os

print("Currently loading the training file for the first session only, to reduce the load on the GPU RAM...")
local_dir = snapshot_download(
    repo_id="gasparyanartur/things-eeg2",
    repo_type="dataset",
    local_dir="things_eeg_raw_sub01",
    allow_patterns=["*sub-01/ses-01/raw_eeg_training.npy"]
)

In [ ]:
import numpy as np

file_path = '/content/things_eeg_raw_sub01/raw-eeg/sub-01/ses-01/raw_eeg_training.npy'
eeg_data = np.load(file_path, allow_pickle=True)

# هون رح نستخدم .item() عشان نستخرج الـ Dictionary اللي جوا المصفوفة
eeg_dict = eeg_data.item()

print(f"Type inside the file: {type(eeg_dict).__name__}")

if isinstance(eeg_dict, dict):
    print("\n=== File contents===")
    for key, value in eeg_dict.items():
        if isinstance(value, np.ndarray):
            print(f"- {key} a matrix with dimensions : {value.shape}")
        elif isinstance(value, list):
            print(f"- {key}: قائمة (List) طولها {len(value)}")
        else:
            print(f"- {key}: {type(value)}")

In [ ]:
import mne
import numpy as np

# 1. استخراج المتغيرات من القاموس اللي اكتشفناه
data = eeg_dict['raw_eeg_data']
ch_names = eeg_dict['ch_names']
ch_types = eeg_dict['ch_types']
sfreq = eeg_dict['sfreq']

print(f"تردد العينة (Sampling Frequency): {sfreq} Hz")
print(f"الفلتر القديم المطبق عالداتا: Highpass={eeg_dict['highpass']}, Lowpass={eeg_dict['lowpass']}")

# 2. بناء هيكل البيانات الخاص بمكتبة MNE
print("\nجاري بناء هيكل MNE...")
# إنشاء معلومات الـ Info اللي بتطلبها المكتبة
info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types=ch_types)

# تحويل مصفوفة Numpy لـ MNE Raw Object
raw = mne.io.RawArray(data, info)

# 3. تطبيق الـ Bandpass Filter (من 1 إلى 40 هرتز) لتنظيف التشويش
print("\nجاري تطبيق Bandpass Filter (من 1 إلى 40 هرتز) لتنقية الإشارة...")
# استخدمنا n_jobs=-1 عشان نخلي الكولاب يستخدم كل المعالجات ويخلص أسرع
raw.filter(l_freq=1.0, h_freq=40.0, n_jobs=-1)


In [ ]:
# 1.  استخراج أسماء القنوات ال64 لتحديد قنات الحدث
channel_names = raw.info['ch_names']

# 2. طباعة الأسماء عشان ندور على قناة الأحداث
print(f"عدد القنوات الكلي: {len(channel_names)}")
print("\n=== أسماء القنوات ===")

for i, name in enumerate(channel_names):
    print(f"{i}: {name}", end=" | ")

    if (i + 1) % 8 == 0:
        print()

In [ ]:
import mne
import numpy as np
import gc  # مكتبة لتنظيف الذاكرة (Garbage Collector)

print("1. Loading data...")
file_path = '/content/things_eeg_raw_sub01/raw-eeg/sub-01/ses-01/raw_eeg_training.npy'
eeg_data = np.load(file_path, allow_pickle=True)
eeg_dict = eeg_data.item()

# بناء الـ MNE Object
info = mne.create_info(ch_names=eeg_dict['ch_names'], sfreq=eeg_dict['sfreq'], ch_types=eeg_dict['ch_types'])
raw = mne.io.RawArray(eeg_dict['raw_eeg_data'], info)

# تنظيف فوري للرام من المصفوفات الخام اللي ما عاد بدنا إياها
del eeg_data, eeg_dict
gc.collect()
print("تم تنظيف الرام بنجاح!")

print("\n2. جاري الفلترة (1 إلى 40 هرتز)...")
raw.filter(l_freq=1.0, h_freq=40.0, n_jobs=-1)

print("\n3. جاري عمل Downsampling لـ 250 هرتز (لحل مشكلة الرام وتسريع التدريب)...")
raw.resample(250)

print("\n4. Epoching)...")
events = mne.find_events(raw, stim_channel='stim')
# استخدمنا preload=True عشان يخزنهم بالرام الجديدة المخففة
epochs = mne.Epochs(raw, events, tmin=-0.2, tmax=0.8, baseline=(None, 0), preload=True)

print("\n=== تمت العملية بنجاح! ===")
print(f"شكل المصفوفة النهائية (عدد الصور، عدد القنوات، طول الإشارة الزمنية المخففة): {epochs.get_data().shape}")

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader
import gc

# 1. استخراج الإشارات كأرقام صافية
print("جاري استخراج المصفوفات...")
eeg_array = epochs.get_data()
labels = epochs.events[:, 2]

# تنظيف الرام قبل التحويل
del epochs
gc.collect()

# 2. التحويل الذكي لـ PyTorch Tensors (بدون Copy)
print("جاري التحويل لـ PyTorch Tensors...")
# استخدام from_numpy بمنع الكراش
X = torch.from_numpy(eeg_array).float()
y = torch.from_numpy(labels).long()

print("\n=== أبعاد الـ Tensors ===")
print(f"بيانات الإدخال (X): {X.shape}")
print(f"الـ (Labels (y: {y.shape}")

# 3. تغليف الداتا بـ DataLoader
batch_size = 64
dataset = TensorDataset(X, y)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

print(f"\nتم تجهيز الـ DataLoader بنجاح! عدد الدفعات (Batches): {len(dataloader)}")

In [ ]:
import torch.nn as nn

# 1. بناء معمارية الـ EEG Encoder
class EEGEncoder(nn.Module):
    def __init__(self, num_channels=64, embed_dim=768):
        super(EEGEncoder, self).__init__()

        # الطبقة الأولى: استخراج الخصائص الزمنية (Temporal Features)
        self.temporal_conv = nn.Sequential(
            nn.Conv1d(in_channels=num_channels, out_channels=128, kernel_size=8, padding=4),
            nn.BatchNorm1d(128),
            nn.GELU(), # دالة التنشيط الحديثة (أحسن من ReLU)
            nn.MaxPool1d(kernel_size=4)
        )

        # الطبقة الثانية: استخراج خصائص أعمق (Spatial-Temporal)
        self.spatial_conv = nn.Sequential(
            nn.Conv1d(in_channels=128, out_channels=256, kernel_size=4, padding=2),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.MaxPool1d(kernel_size=4)
        )

        # تجميع المتوسطات عشان نثبت الحجم
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.flatten = nn.Flatten()

        # Projector: الطبقة اللي بتجبر الدماغ يحكي نفس لغة CLIP
        self.projector = nn.Sequential(
            nn.Linear(256, 512),
            nn.GELU(),
            nn.Linear(512, embed_dim) # هون بنطلع الـ 768 رقم
        )

    def forward(self, x):
        # x بكون شكلها (Batch, 64, 251)
        x = self.temporal_conv(x)
        x = self.spatial_conv(x)
        x = self.pool(x)       # بصفي الحجم (Batch, 256, 1)
        x = self.flatten(x)    # بنفردها لـ (Batch, 256)
        out = self.projector(x) # بنحولها لـ (Batch, 768)
        return out

# 2. إنشاء نسخة من الموديل
model = EEGEncoder()
print("تم بناء الموديل (EEG Encoder) بنجاح!")

# 3. رح نسحب دفعة واحدة (Batch) من الداتا تبعتنا ونجربها عالموديل
print("\nجاري اختبار الموديل على أول دفعة من البيانات...")
X_batch, y_batch = next(iter(dataloader)) # سحبنا 64 إشارة

# تمرير الإشارات جوا الموديل
output_embeddings = model(X_batch)

print("\n=== The result after the signal has entered the model ===")
print(f"شكل الداتا لما دخلت (X_batch): {X_batch.shape}")
print(f"شكل الداتا لما طلعت (Output): {output_embeddings.shape}")

In [ ]:
import torch.nn.functional as F

# 1. بناء دالة الخسارة InfoNCE Loss
class ContrastiveLoss(nn.Module):
    def __init__(self, temperature=0.1):
        super().__init__()
        # الـ temperature (الحرارة) هو رقم بيتحكم بمدى قسوة الموديل في التفريق بين الصور
        self.temperature = temperature

    def forward(self, eeg_embeddings, image_embeddings):
        # الخطوة 1: تطبيع (Normalize) المتجهات عشان نقيس التشابه صح (Cosine Similarity)
        eeg_embeddings = F.normalize(eeg_embeddings, dim=-1)
        image_embeddings = F.normalize(image_embeddings, dim=-1)

        # الخطوة 2: حساب مصفوفة التشابه (Similarity Matrix)
        # النتيجة رح تكون مصفوفة 64x64 بتعبر عن نسبة التشابه بين كل إشارة وكل صورة بالدفعة
        logits = torch.matmul(eeg_embeddings, image_embeddings.T) / self.temperature

        # الخطوة 3: بناء الـ Labels الصحيحة
        # بما إن الإشارة رقم 0 تابعة للصورة رقم 0، والإشارة 1 للصورة 1...
        # إذن الإجابات الصحيحة لازم تكون على الخط القطري (Diagonal) للمصفوفة
        labels = torch.arange(logits.size(0)).to(logits.device)

        # الخطوة 4: حساب نسبة الخسارة (CrossEntropy)
        # بنحسبها مرة من جهة الـ EEG ومرة من جهة الصور وبناخذ المتوسط (عشان التوازن)
        loss_eeg = F.cross_entropy(logits, labels)
        loss_image = F.cross_entropy(logits.T, labels)

        return (loss_eeg + loss_image) / 2

# 2. اختبار دالة الخسارة اللي بناها العضو الثاني
loss_fn = ContrastiveLoss()

# بما إنه لسا ما حملنا الصور الحقيقية، رح نصنع "متجهات صور وهمية" (كأنها طالعة من CLIP) بس عشان نفحص الكود
print("Generating dummy image vectors of size (64, 768) for testing...")
dummy_image_embeddings = torch.randn(64, 768)

# 3. حساب نسبة الخطأ (Loss) بين مخرجات الدماغ والصور
loss = loss_fn(output_embeddings, dummy_image_embeddings)

print("\n=== نتيجة فحص دالة الخسارة (InfoNCE) ===")
print(f"قيمة الخسارة (Loss) المبدئية: {loss.item():.4f}")

In [ ]:
import torch.optim as optim

# 1. تعريف المُحسّن (Optimizer)
# Learning Rate (lr) = 0.001 (سرعة التعلم)
optimizer = optim.AdamW(model.parameters(), lr=0.001)

print("=== اختبار قدرة الموديل على التعلم (Training Loop) ===")
print("رح ندرب الموديل 10 مرات على نفس الدفعة عشان نشوف إذا الـ Loss بينزل...\n")

# 2. حلقة التدريب (Training Loop)
for epoch in range(1, 11): # 10 لفات

    # أ. تصفير الذاكرة تبعت المُحسّن (عشان ما يراكم الأخطاء القديمة)
    optimizer.zero_grad()

    # ب. التنبؤ (Forward Pass): ندخل الإشارات عالموديل
    output_embeddings = model(X_batch)

    # ج. حساب الخسارة (Calculate Loss)
    # تذكر: بالحقيقة هون رح نستخدم متجهات CLIP للصور الحقيقية بدل الوهمية
    loss = loss_fn(output_embeddings, dummy_image_embeddings)

    # د. التعلم من الخطأ (Backward Pass): حساب التفاضلات
    loss.backward()

    # هـ. تحديث الأوزان (Update Weights)
    optimizer.step()

    # طباعة النتيجة
    print(f"اللفة (Epoch) رقم {epoch}: قيمة الخسارة (Loss) = {loss.item():.4f}")


In [ ]:
# 1. تنزيل مكتبات توليد الصور
!pip install diffusers accelerate

from diffusers import StableDiffusionPipeline
import torch
import matplotlib.pyplot as plt

# 2. تحميل نموذج Stable Diffusion 1.5 (النسخة الخفيفة)
print("جاري تحميل Stable Diffusion ...")
device = "cuda" if torch.cuda.is_available() else "cpu"
pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
)
pipe = pipe.to(device)

# إيقاف فحص السلامة (Safety Checker) عشان ما يعطينا شاشات سودا بالغلط
pipe.safety_checker = None

# 3. محاكاة توليد صورة من إشارة دماغ!
print("\nPreparing the brain signal for image generation...")

# رح ناخذ أول إشارة من اللي دربناهم قبل شوي
# حجمها كان (768)
single_brain_vector = output_embeddings[0].detach()

# حيلة العضو الثالث: تكرار الإشارة 77 مرة عشان يفهمها Stable Diffusion كأنها Prompt
# الحجم بصير (1, 77, 768)
brain_prompt_embeds = single_brain_vector.unsqueeze(0).unsqueeze(0).repeat(1, 77, 1)
brain_prompt_embeds = brain_prompt_embeds.to(device, dtype=pipe.dtype)

# 4. توليد الصورة!
print("Currently drawing the image directly from the brain... ")
# بنمرر المصفوفة تبعتنا لـ prompt_embeds بدل ما نكتب prompt نصي
image = pipe(prompt_embeds=brain_prompt_embeds, num_inference_steps=20).images[0]

# 5. عرض النتيجة
plt.figure(figsize=(6, 6))
plt.imshow(image)
plt.axis('off')
plt.title("Image generated from brain signals (experimental)")
plt.show()